# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row =** one page, on one day (from `fact_content_daily_performance`).
**Time window:** `month=2026-03` — a mid-panel month, since the guide warns
never to develop label logic on the `_sample` table (the sealed final month,
June 2026).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
unit_of_analysis = "one page, one day"
time_window = "month=2026-03"
print(f"Unit of analysis: {unit_of_analysis}")
print(f"Time window: {time_window}")

Unit of analysis: one page, one day
Time window: month=2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature (knowable at decision time):** impressions, avg_position, clicks, ctr
**Label/proxy:** a decline signal built from this month's own trend
**Context (background, not fed to the model):** client id (pseudonymized), report_date
**Excluded, and why:** raw query text, raw URLs — not public-safe; any
future-window metric — would leak the label

In [2]:
fields = {
    "feature": ["gsc_impressions", "gsc_avg_position", "gsc_clicks", "ctr (derived)"],
    "label_proxy": ["is_declining (built from ctr trend this month)"],
    "context": ["content_hash_id (pseudonymized)", "client_hash_id (pseudonymized)", "report_date"],
    "excluded": ["raw query text", "raw URLs", "any future-window metric"],
}
for k, v in fields.items():
    print(f"{k}: {v}")

feature: ['gsc_impressions', 'gsc_avg_position', 'gsc_clicks', 'ctr (derived)']
label_proxy: ['is_declining (built from ctr trend this month)']
context: ['content_hash_id (pseudonymized)', 'client_hash_id (pseudonymized)', 'report_date']
excluded: ['raw query text', 'raw URLs', 'any future-window metric']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{hf_token}');")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet"

Query 1: verifying grain — one row really is one page-day.

In [4]:
q1 = con.execute(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT content_hash_id || '|' || report_date) AS unique_page_days
    FROM read_parquet('{FACT_MARCH}')
""").df()
print(q1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_page_days
0     9841378           9841378


In [5]:
q2 = con.execute(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{FACT_MARCH}')
""").df()
print(q2)

   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


In [6]:
q3 = con.execute(f"""
    SELECT COUNT(*) AS available_rows
    FROM read_parquet('{FACT_MARCH}')
    WHERE gsc_data_available IS TRUE
""").df()
total = q2['row_count'][0]
print(q3)
print(f"Available: {q3['available_rows'][0]} / {total} ({q3['available_rows'][0]/total:.1%})")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   available_rows
0         3611061
Available: 3611061 / 9841378 (36.7%)


In [7]:
features_df = con.execute(f"""
    SELECT content_hash_id, report_date, gsc_impressions, gsc_avg_position, gsc_clicks,
           CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE NULL END AS ctr
    FROM read_parquet('{FACT_MARCH}')
    WHERE gsc_data_available IS TRUE
    LIMIT 1000
""").df()
features_df.head()

,content_hash_id,report_date,gsc_impressions,gsc_avg_position,gsc_clicks,ctr
0,content_b7e512995f79d5a6,2026-03-01,20,3.350000,0,0.000
1,content_05597932fe4da067,2026-03-01,1,0.000000,0,0.000
2,content_7a105f548d9c6916,2026-03-01,125,4.928000,1,0.008
3,content_905aa32a0230694e,2026-03-01,7,4.000000,0,0.000
4,content_a3ea9792f793ec72,2026-03-01,11,2.272727,0,0.000


- **gsc_impressions** — available when? Same-day Search Console log.
- **gsc_avg_position** — available when? Same-day ranking snapshot.
- **gsc_clicks** — available when? Same-day click log.
- **ctr** — available when? Derived same-day from clicks/impressions.
- **report_date** — available when? Always known, it's the row's own date.

**The leakage trap:** I'll build a label from `ctr`, then "test" a feature that
is itself `ctr` — the correlation will look almost perfect, because the label
was built from it. That's not a real signal, it's leakage. Then I'll switch to
an honest, independent feature and show the real (weaker) number.

In [8]:
print(features_df["ctr"].describe())
print(features_df["ctr"].value_counts().head(10))

count    1000.000000
mean        0.002779
std         0.032540
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: ctr, dtype: float64
ctr
0.000000    831
0.015873      2
0.012500      2
0.002994      2
0.007692      2
0.006329      2
0.001988      2
0.007519      2
0.004310      2
0.008000      1
Name: count, dtype: int64


In [9]:
print("Rows total:", len(features_df))
print("NaN ctr count:", features_df["ctr"].isna().sum())

Rows total: 1000
NaN ctr count: 0


In [10]:
# Build the clean dataframe (drop NaNs, define the label)
clean_df = features_df.dropna(subset=["ctr", "gsc_avg_position"]).copy()
clean_df["is_declining"] = (clean_df["ctr"] == 0).astype(int)

# The trap: can a "model" trivially get near-perfect using the leaked feature?
# Rule: predict is_declining whenever ctr == 0 (this is EXACTLY how the label was built)
leaked_prediction = (clean_df["ctr"] == 0).astype(int)
leak_accuracy = (leaked_prediction == clean_df["is_declining"]).mean()
print(f"'Model' accuracy using LEAKED feature (ctr==0 rule): {leak_accuracy:.3f}")

# Now the honest version: use a feature that does NOT define the label
honest_threshold = clean_df["gsc_avg_position"].median()
honest_prediction = (clean_df["gsc_avg_position"] > honest_threshold).astype(int)
honest_accuracy = (honest_prediction == clean_df["is_declining"]).mean()
print(f"Honest accuracy using INDEPENDENT feature (avg_position rule): {honest_accuracy:.3f}")

'Model' accuracy using LEAKED feature (ctr==0 rule): 1.000
Honest accuracy using INDEPENDENT feature (avg_position rule): 0.595


Leaked feature (ctr==0) gives perfect accuracy (1.000) because it literally
defines the label — that's the leak. Removing it and using an independent
feature (avg_position) drops accuracy to a realistic 0.595, barely above
chance — showing how much of the earlier "signal" was fake.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice can't tell me client identity, revenue impact, or why a page
declined (seasonality vs. a real quality drop). Only 36.7% of March page-days
have gsc_data_available == TRUE — meaning nearly two-thirds of rows can't be
used for GSC-based features on a given day without filtering first. Any
GSC-based feature must filter on gsc_data_available IS TRUE or risk training
on mostly-missing data.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Named limitation: cannot infer cause of decline (seasonality vs. quality) "
      "from this slice alone — only that decline is observed.")
print(f"Named limitation: only 36.7% of page-days have gsc_data_available == TRUE.")

Named limitation: cannot infer cause of decline (seasonality vs. quality) from this slice alone — only that decline is observed.
Named limitation: only 36.7% of page-days have gsc_data_available == TRUE.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.